# Ch.3 — Data Validation & Drift Detection

> **Mission**: Build validation firewalls that catch distribution shift BEFORE production deployment.
>
> **The RealtyML Crisis (Final Chapter)**: Sarah has cleaned data (Ch.1) and rebalanced classes (Ch.2), but Portland MAE is still 128k (target: <95k). This notebook reveals the final failure: California training data (MedInc mean $38k) doesn't match Portland production data (MedInc mean $52k). Had validation existed, the 174k MAE disaster would never have reached production.
>
> **What you'll build**: (1) Great Expectations suite validating schema + distributions, (2) KS test detecting distribution shift, (3) Drift alert system catching the California → Portland shift, (4) Quantify impact: 128k → 89k MAE by addressing drift.

**Execution time**: ~4 minutes (tested on California Housing 20,640 samples)

---

## Setup — Load Libraries

We'll use:
- **Great Expectations**: Declarative schema + distribution validation
- **scipy.stats**: Kolmogorov-Smirnov test for distribution comparison
- **sklearn**: California Housing data + LinearRegression baseline
- **matplotlib**: Visualize distribution shifts

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Import the required libraries: great_expectations, matplotlib.pyplot, numpy, pandas, scipy.stats, sklearn.datasets
# 2. Set random seed (np.random.seed) and configure the plot style
#
# Hint:
#   import pandas as pd
#   import numpy as np
#   import matplotlib.pyplot as plt

---

## Part 1: Load California Training Data

This is the data RealtyML originally trained on — 20,640 California homes from the 1990 census.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Call fetch_california_housing() to load the dataset object
# 2. Create a pandas DataFrame: pd.DataFrame(housing.data, columns=housing.feature_names)
# 3. Append target column: df['MedHouseVal'] = housing.target
# 4. Print dataset shape and preview with df.head()
#
# Hint:
#   housing = fetch_california_housing()
#   df = pd.DataFrame(housing.data, columns=housing.feature_names)
#   df['MedHouseVal'] = housing.target

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Call `mean()` to produce the result
#
# Hint:
#    # implement using the APIs described above

---

## Part 2: Simulate Portland Production Data (Distribution Shift)

**Sarah's discovery**: Portland's median income is 37% higher than California's. We'll simulate this shift by scaling `MedInc`.

**Real-world context**: This mirrors the actual California → Portland demographic difference (2020 census data shows Portland metro area median household income $83k vs California statewide $75k, but the housing-specific gap is larger due to Portland's tech worker concentration).

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Create df_portland = df_california.copy()
# 2. Scale MedInc by 1.37 to simulate the income shift
# 3. Scale MedHouseVal by 1.15 to reflect higher Portland prices
# 4. Print comparison of California vs Portland mean income
#
# Hint:
#   df_portland = df_california.copy()
#   df_portland['MedInc'] = df_portland['MedInc'] * ???  # 37% increase
#   df_portland['MedHouseVal'] = df_portland['MedHouseVal'] * ???

### Visualize Distribution Shift

Side-by-side histograms showing California (training) vs Portland (production) income distributions.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Create overlaid histograms: one for California, one for Portland
# 2. Add vertical axvline for each distribution's mean with label
# 3. Style with dark background; set xlabel, ylabel, title
# 4. Save to img/<name>-distribution-shift.png
#
# Hint:
#   ax.hist(df_california['MedInc'], bins=50, alpha=0.6, label='California')
#   ax.hist(df_portland['MedInc'],   bins=50, alpha=0.6, label='Portland')
#   ax.axvline(ca_mean, color='#3b82f6', linestyle='--', label=f'CA Mean: {ca_mean:.2f}')

---

## Part 3: Great Expectations — Schema & Distribution Validation

**Goal**: Build a declarative validation suite that checks:
1. **Schema validation**: Types, ranges, non-null constraints
2. **Distribution validation**: Mean, std within expected bounds (based on training data)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Call ge.from_pandas(df_california) to wrap the DataFrame
# 2. Add range expectations: expect_column_values_to_be_between(col, min_value=???, max_value=???)
# 3. Add mean expectations: expect_column_mean_to_be_between(col, min_value=ca_mean*0.9, max_value=ca_mean*1.1)
# 4. Add null check: expect_column_values_to_not_be_null(col)
# 5. Test on Portland data to confirm failures are detected
#
# Hint:
#   df_ge = ge.from_pandas(df_california)
#   df_ge.expect_column_values_to_be_between('MedInc', min_value=???, max_value=???)
#   df_ge.expect_column_mean_to_be_between('MedInc', min_value=ca_mean*0.9, max_value=ca_mean*1.1)
#   df_ge.expect_column_values_to_not_be_null('MedInc')

### Test Validation Suite on Portland Data

**Critical test**: Does the validation suite catch the California → Portland distribution shift?

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Call ge.from_pandas(df_california) to wrap the DataFrame
# 2. Add range expectations: expect_column_values_to_be_between(col, min_value=???, max_value=???)
# 3. Add mean expectations: expect_column_mean_to_be_between(col, min_value=ca_mean*0.9, max_value=ca_mean*1.1)
# 4. Add null check: expect_column_values_to_not_be_null(col)
# 5. Test on Portland data to confirm failures are detected
#
# Hint:
#   df_ge = ge.from_pandas(df_california)
#   df_ge.expect_column_values_to_be_between('MedInc', min_value=???, max_value=???)
#   df_ge.expect_column_mean_to_be_between('MedInc', min_value=ca_mean*0.9, max_value=ca_mean*1.1)
#   df_ge.expect_column_values_to_not_be_null('MedInc')

---

## Part 4: Kolmogorov-Smirnov Test — Statistical Distribution Comparison

**Goal**: Use KS test to detect distribution shift across all features.

**Interpretation**: p-value < 0.05 means distributions are significantly different (drift detected).

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Loop over feature columns
# 2. Call ks_2samp(df_california[col], df_portland[col]) for each
# 3. Store (statistic, p_value) results
# 4. Flag features where p_value < 0.05 as significantly shifted
# 5. Print a summary table of all features
#
# Hint:
#   from scipy.stats import ks_2samp
#   stat, p_value = ks_2samp(df_california[col], df_portland[col])
#   if p_value < 0.05: print(f'{col}: DRIFT DETECTED (p={p_value:.4f})')

### Visualize KS Test Results

Bar chart showing p-values for all features. Red bars (p < 0.05) indicate drift.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Use subplots(???) to implement this step
# 2. Use set_facecolor(???) to implement this step
# 3. Use bar(???) to implement this step
#
# Hint:
#   # see solution cell for the required API calls

---

## Part 5: Custom Drift Alert System

**Goal**: Implement a simple production-ready drift monitoring function.

**Logic**: Alert if mean shifts by >15% OR KS test p-value < 0.05.

In [ ]:
def check_drift_alert(train_data, prod_data, feature, mean_threshold_pct=15, p_value_threshold=0.05):
    """
    TODO #9: Implement `check_drift_alert()`.

    Steps:
    1. Call dataset(???) to implement this step
    2. Call mean(???) to implement this step
    3. Call check_drift_alert(???) to implement this step

    Hint:
        # see solution cell for the required API calls

    Returns: {
    """
    raise NotImplementedError("TODO #9: implement check_drift_alert()")

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Use 15(???) to implement this step
# 2. Use check_drift_alert(???) to implement this step
# 3. Use Critical(???) to implement this step
#
# Hint:
#   # see solution cell for the required API calls

---

## Part 6: Quantify Impact on Model Performance

**Goal**: Show how distribution shift translates to MAE degradation.

**Experiment**:
1. Train on California data
2. Test on California (baseline MAE)
3. Test on Portland (MAE with drift)
4. Retrain on combined data (drift correction)
5. Test on Portland again (MAE after correction)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Call train_test_split(X, y, test_size=0.2, random_state=SEED)
# 2. Print resulting train/test shapes
#
# Hint:
#   X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=???, random_state=???)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Use LinearRegression(???) to implement this step
# 2. Use fit(???) to implement this step
# 3. Use predict(???) to implement this step
#
# Hint:
#   model_baseline = LinearRegression(???)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Call train_test_split(X, y, test_size=0.2, random_state=SEED)
# 2. Print resulting train/test shapes
#
# Hint:
#   X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=???, random_state=???)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Use samples(???) to implement this step
# 2. Use concat(???) to implement this step
# 3. Use data(???) to implement this step
#
# Hint:
#   model_corrected = LinearRegression(???)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Instantiate SMOTE(random_state=42, k_neighbors=5)
# 2. Call smote.fit_resample(X_train, y_train_class) to generate synthetic minority samples
# 3. Print class distribution before and after resampling
#
# Hint:
#   smote = SMOTE(random_state=42, k_neighbors=???)
#   X_train_smote, y_train_smote = smote.fit_resample(???, ???)
#   pd.Series(y_train_smote).value_counts()

---

## Part 7: Production Validation Pipeline Summary

**What you built**:
1. **Great Expectations suite**: 7 expectations validating schema + distributions
2. **KS test**: Statistical distribution comparison (detected MedInc drift, p ≈ 0)
3. **Drift alert system**: Custom function flagging >15% mean shifts
4. **Impact quantification**: Drift → $X k MAE increase, correction → $Y k improvement

**Production deployment checklist**:
- [ ] Define expectations on training data (schema + distribution bounds)
- [ ] Run KS tests on all features (batch or streaming)
- [ ] Set alert thresholds (mean shift >15%, p-value < 0.05)
- [ ] Create SOP for drift response:
 - Minor drift (10-20% shift): Log warning, monitor
 - Major drift (>30% shift): Block deployment, page on-call
- [ ] Automate: Run validation on every production batch
- [ ] Dashboard: Track drift metrics over time (mean, std, p-values)

**Sarah's final quote**:
> "We need a firewall between bad data and production models. This validation suite would have caught the California → Portland shift before deployment. Now it's part of our CI/CD pipeline — no model ships without passing drift checks."

---

## Conclusion & Next Steps

### What You Learned

1. **Distribution shift is silent but deadly**: Models fail on out-of-distribution data even if the data is "valid"
2. **Validation has two layers**: Schema (types, ranges) + Distribution (mean, std, shape)
3. **KS test is your friend**: Simple statistical test detecting distribution differences (p < 0.05 = drift)
4. **Prevention > Reaction**: Catching drift BEFORE deployment saves MAE degradation

### Bridge to ML Track

You've now completed the **Data Fundamentals** track:
- Ch.1: Outlier detection, imputation, EDA
- Ch.2: Class imbalance (SMOTE, class weights)
- Ch.3: Data validation, drift detection

**You're ready for [01_regression/ch01_linear_regression](../../01_regression/ch01_linear_regression/README.md)**. When you see the California Housing dataset again, you'll automatically ask:
- Are there outliers? (Ch.1 habit)
- Are classes balanced? (Ch.2 habit)
- Does my test set match production? (Ch.3 habit)

You've built the **#1 skill separating production ML engineers from Kaggle competitors**: validating data BEFORE training models.

### Real-World Next Steps

1. **Add to your ML projects**: Insert validation checks in every data pipeline
2. **Practice drift detection**: Run KS tests on any A/B test, seasonal data, or geographic split
3. **Build a portfolio project**: "Automated Drift Monitoring Dashboard" using Evidently AI
4. **Interview prep**: Practice explaining "what is distribution shift?" in 60 seconds

---

** Grand Challenge Complete: RealtyML Production System Fixed!**

Sarah presents to the board:
- Constraint #1 (Garbage In): Outliers removed, proper imputation → cleaned data
- Constraint #2 (Imbalance Blindness): SMOTE + class weights → balanced training
- Constraint #3 (Drift Ignored): Great Expectations + KS tests → drift detection firewall
- **Portland MAE: 174k → 89k (target: <95k) achieved!**

The product is saved. The board approves Series B funding. Sarah gets promoted to Principal Data Scientist.

**Her final commit message**: `feat: add drift detection to deployment pipeline — never again `